In [ ]:
# ===== CELL 1: Setup (run this first, every time you open the notebook) =====
import os, sys, re, io, json, glob, math, time, shutil, zipfile, subprocess, importlib.util
import numpy as np, pandas as pd, soundfile as sf, requests, torch
from scipy.signal import resample_poly
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')

# Results that must survive (scores, metrics, splits) go to Drive. Share this folder with the team.
OUT_DIR = '/content/drive/MyDrive/CMPE258_Deepfake/shriya_aasist'
for sub in ['scores/parts', 'keys', 'metrics', 'splits', 'dfadd']:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

# Temporary space on the Colab machine (wiped when the session ends; that's fine)
DATA_DIR = '/content/asvspoof5_data'   # protocol files
WORK_DIR = '/content/work'             # each tar is downloaded + extracted here, scored, then deleted
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

ZENODO_URL = 'https://zenodo.org/records/14498691/files'                 # ASVspoof 5
DFADD_URL = 'https://huggingface.co/datasets/isjwdu/DFADD/resolve/main'  # DFADD (updated Apr 2025 zips)
REPO_DIR = '/content/asvspoof5'                                          # official code + AASIST weights

DEV_TARS = [f'flac_D_{s}.tar' for s in ['aa', 'ab', 'ac']]
EVAL_TARS = [f'flac_E_{s}.tar' for s in ['aa', 'ab', 'ac', 'ad', 'ae', 'af', 'ag', 'ah', 'ai', 'aj']]
TRAIN_TARS = [f'flac_T_{s}.tar' for s in ['aa', 'ab', 'ac', 'ad', 'ae']]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then rerun this cell.')
print('Free local disk: %.0f GB' % (shutil.disk_usage('/content').free / 1e9))

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/asvspoof-challenge/asvspoof5.git', REPO_DIR], check=True)
print('Official ASVspoof 5 repo ready.')

Mounted at /content/drive
GPU: Tesla T4
Free local disk: 75 GB
Official ASVspoof 5 repo ready.


In [ ]:
# ===== CELL 2: Helper functions + load the official AASIST model =====
def download(url, dest, retries=5):
    """Download with a progress line. If it gets cut off, rerunning continues where it stopped."""
    for attempt in range(1, retries + 1):
        have = os.path.getsize(dest) if os.path.exists(dest) else 0
        headers = {'Range': f'bytes={have}-'} if have else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=60) as r:
                if r.status_code == 416:          # file was already fully downloaded
                    return dest
                r.raise_for_status()
                if r.status_code == 200:          # server sent the whole file, start over
                    have = 0
                total = have + int(r.headers.get('Content-Length', 0))
                done, last = have, 0.0
                with open(dest, 'ab' if have else 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8 << 20):
                        f.write(chunk)
                        done += len(chunk)
                        if total and time.time() - last > 5:
                            print(f'\r  {os.path.basename(dest)}: {100 * done / total:5.1f}% ({done >> 20} MB)', end='')
                            last = time.time()
            if total and os.path.getsize(dest) != total:
                raise IOError('download incomplete')
            print(f'\r  {os.path.basename(dest)}: downloaded ({os.path.getsize(dest) >> 20} MB)          ')
            return dest
        except Exception as e:
            print(f'\n  download problem ({e}); retry {attempt}/{retries}')
            time.sleep(10)
    raise RuntimeError(f'Could not download {url}')


def extract_tar(tar_path, dest):
    os.makedirs(dest, exist_ok=True)
    subprocess.run(['tar', '-xf', tar_path, '-C', dest], check=True)


def find_audio(root, exts=('.flac', '.wav')):
    """Map file ID (name without extension) -> full path, wherever the tar put it."""
    found = {}
    for dirpath, _, files in os.walk(root):
        for fn in files:
            if fn.lower().endswith(exts):
                found[os.path.splitext(fn)[0]] = os.path.join(dirpath, fn)
    return found


def save_tsv(df, path):
    """Write to a temp file first so a crash never leaves a half-written file that looks finished."""
    tmp = path + '.tmp'
    df.to_csv(tmp, sep='\t', index=False)
    os.replace(tmp, path)


CUT = 64600  # official AASIST input: 64,600 samples (~4 s at 16 kHz)


class ScoringDataset(Dataset):
    """Based on my ASVspoof5Dataset, changed to load audio exactly like the official AASIST TestDataset:
    first 64,600 samples, short clips are REPEATED (not zero-padded), and it returns the file ID."""

    def __init__(self, ids, paths):
        self.ids, self.paths = list(ids), list(paths)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        x, sr = sf.read(self.paths[i], dtype='float32')
        if x.ndim > 1:
            x = x.mean(axis=1)                        # stereo -> mono
        assert sr == 16000, f'{self.paths[i]} is {sr} Hz, expected 16000'
        if len(x) == 0:
            x = np.zeros(CUT, dtype='float32')
        elif len(x) >= CUT:
            x = x[:CUT]
        else:
            x = np.tile(x, CUT // len(x) + 1)[:CUT]
        return torch.from_numpy(np.ascontiguousarray(x)), self.ids[i]


def load_aasist():
    base = os.path.join(REPO_DIR, 'Baseline-AASIST')
    spec = importlib.util.spec_from_file_location('aasist_model', os.path.join(base, 'models', 'AASIST.py'))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    config = json.load(open(os.path.join(base, 'config', 'AASIST_ASVspoof5.conf')))['model_config']
    net = module.Model(config).to(device)
    weights = os.path.join(base, 'models', 'weights', 'AASIST', 'best.pth')
    net.load_state_dict(torch.load(weights, map_location=device))
    return net.eval()


model = load_aasist()
print('Loaded official ASVspoof 5 AASIST checkpoint:', sum(p.numel() for p in model.parameters()), 'parameters')


@torch.no_grad()
def score(ids, paths, batch_size=64):
    loader = DataLoader(ScoringDataset(ids, paths), batch_size=batch_size, shuffle=False,
                        num_workers=min(4, os.cpu_count() or 1), pin_memory=(device == 'cuda'))
    names, logits = [], []
    for x, keys in tqdm(loader, leave=False):
        _, out = model(x.to(device))
        names += list(keys)
        logits.append(out.float().cpu().numpy())
    L = np.concatenate(logits) if logits else np.zeros((0, 2))
    # Official convention: score = bona fide logit (column 1). Higher = more likely real.
    return pd.DataFrame({'filename': names, 'cm-score': L[:, 1],
                         'logit_spoof': L[:, 0], 'logit_bonafide': L[:, 1]})


def part_path(prefix, tar_name):
    return os.path.join(OUT_DIR, 'scores', 'parts', f'{prefix}_{tar_name[:-4]}.tsv')


def run_tar_job(prefix, tar_name, wanted_ids):
    """One tar at a time: download -> extract -> score the files we need -> save scores to Drive -> delete audio."""
    out = part_path(prefix, tar_name)
    if os.path.exists(out):
        print(f'[skip] {tar_name} already scored for {prefix}')
        return
    if shutil.disk_usage('/content').free < 20e9:
        print('WARNING: under 20 GB free on the Colab disk; this tar may not fit.')
    tar_path = os.path.join(WORK_DIR, tar_name)
    ex_dir = tar_path + '_extracted'
    for attempt in (1, 2):
        download(f'{ZENODO_URL}/{tar_name}', tar_path)
        try:
            extract_tar(tar_path, ex_dir)
            break
        except subprocess.CalledProcessError:
            print('  tar file looks damaged, downloading it again')
            os.remove(tar_path)
            shutil.rmtree(ex_dir, ignore_errors=True)
    else:
        raise RuntimeError(f'{tar_name} failed twice')
    os.remove(tar_path)                                   # free disk before scoring
    audio = find_audio(ex_dir)
    todo = sorted(k for k in audio if k in wanted_ids)
    print(f'  {tar_name}: {len(audio)} audio files, scoring {len(todo)}')
    save_tsv(score(todo, [audio[k] for k in todo]), out)
    shutil.rmtree(ex_dir)
    print(f'  saved {os.path.basename(out)} to Drive')


def merge_parts(prefix, tars, expected_ids, out_name):
    parts = [part_path(prefix, t) for t in tars]
    not_done = [os.path.basename(p) for p in parts if not os.path.exists(p)]
    if not_done:
        raise RuntimeError(f'Not finished yet: {not_done}. Rerun the cell above.')
    df = pd.concat([pd.read_csv(p, sep='\t', dtype={'filename': str}) for p in parts])
    df = df.drop_duplicates('filename')
    df = df[df.filename.isin(expected_ids)].reset_index(drop=True)
    missing = sorted(set(expected_ids) - set(df.filename))
    print(f'{prefix}: scored {len(df)} of {len(expected_ids)} expected files; missing {len(missing)}')
    if missing:
        print('  first missing IDs:', missing[:10])
    save_tsv(df, os.path.join(OUT_DIR, 'scores', out_name))
    return df


# Official ASVspoof 5 metrics (minDCF is the primary metric)
sys.path.insert(0, os.path.join(REPO_DIR, 'evaluation-package'))
from calculate_metrics import calculate_minDCF_EER_CLLR_actDCF

METRICS_FILE = os.path.join(OUT_DIR, 'metrics', 'aasist_metrics.json')


def metrics(scores_df, keys_df, name):
    """keys_df needs columns 'filename' and 'KEY' (bonafide/spoof)."""
    m = scores_df.merge(keys_df[['filename', 'KEY']], on='filename')
    if m.KEY.nunique() < 2:
        print(f'{name}: needs both bona fide and spoof files, skipping')
        return None
    mindcf, eer, cllr, actdcf = calculate_minDCF_EER_CLLR_actDCF(
        m['cm-score'].to_numpy(), m['KEY'].to_numpy(), output_file=os.devnull, printout=False)
    res = {'set': name, 'n_files': len(m), 'n_bonafide': int((m.KEY == 'bonafide').sum()),
           'minDCF': round(float(mindcf), 4), 'EER_%': round(100 * float(eer), 2),
           'CLLR': round(float(cllr), 4), 'actDCF': round(float(actdcf), 4)}
    allm = json.load(open(METRICS_FILE)) if os.path.exists(METRICS_FILE) else {}
    allm[name] = res
    json.dump(allm, open(METRICS_FILE, 'w'), indent=2)
    print(res)
    return res

Loaded official ASVspoof 5 AASIST checkpoint: 297866 parameters


In [ ]:
# ===== CELL 3: Protocol files (labels for every file) + integrity check =====
COLS = ['SPEAKER_ID', 'FLAC_FILE_NAME', 'SPEAKER_GENDER', 'CODEC', 'CODEC_Q',
        'CODEC_SEED', 'ATTACK_TAG', 'ATTACK_LABEL', 'KEY', 'TMP']
PROTO_FILES = {'train': 'ASVspoof5.train.tsv', 'dev': 'ASVspoof5.dev.track_1.tsv',
               'eval': 'ASVspoof5.eval.track_1.tsv'}
EXPECTED_ROWS = {'train': 182357, 'dev': 140950, 'eval': 680774}


def find_proto(name):
    hits = glob.glob(os.path.join(DATA_DIR, '**', name), recursive=True)
    return hits[0] if hits else None


if not all(find_proto(f) for f in PROTO_FILES.values()):
    tgz = download(f'{ZENODO_URL}/ASVspoof5_protocols.tar.gz', os.path.join(WORK_DIR, 'ASVspoof5_protocols.tar.gz'))
    extract_tar(tgz, DATA_DIR)
    os.remove(tgz)

print('--- ASVspoof 5 PROTOCOL CHECK ---')
proto = {}
for split, fn in PROTO_FILES.items():
    path = find_proto(fn)
    print(f'[{"EXISTS" if path else "MISSING"}] {fn}')
    assert path, f'{fn} is missing'
    df = pd.read_csv(path, sep=r'\s+', header=None, names=COLS, dtype=str)
    df['filename'] = df['FLAC_FILE_NAME']
    assert set(df.KEY.unique()) <= {'bonafide', 'spoof'}, f'unexpected KEY values in {fn}: {df.KEY.unique()[:5]}'
    flag = 'OK' if len(df) == EXPECTED_ROWS[split] else f'expected {EXPECTED_ROWS[split]}'
    print(f'    {split}: {len(df)} files ({flag}), {(df.KEY == "bonafide").sum()} bona fide, '
          f'{(df.KEY == "spoof").sum()} spoof, {df.SPEAKER_ID.nunique()} speakers')
    # key file in the official evaluation-package format (for Smitha's harness)
    save_tsv(df[['filename', 'KEY']].rename(columns={'KEY': 'cm-label'}),
             os.path.join(OUT_DIR, 'keys', f'{split}_cm_keys.tsv'))
    proto[split] = df
shutil.copy(find_proto(PROTO_FILES['eval']), os.path.join(OUT_DIR, 'keys'))  # full eval protocol, for the slices
print('Protocols loaded. Key files saved to Drive.')

  ASVspoof5_protocols.tar.gz: downloaded (19 MB)          
--- ASVspoof 5 PROTOCOL CHECK ---
[EXISTS] ASVspoof5.train.tsv
    train: 182357 files (OK), 18797 bona fide, 163560 spoof, 400 speakers
[EXISTS] ASVspoof5.dev.track_1.tsv
    dev: 140950 files (OK), 31334 bona fide, 109616 spoof, 785 speakers
[EXISTS] ASVspoof5.eval.track_1.tsv
    eval: 680774 files (OK), 138688 bona fide, 542086 spoof, 737 speakers
Protocols loaded. Key files saved to Drive.


In [ ]:
# ===== CELL 4: Score the DEV set (3 tars, ~20 GB). Rerun if it stops; finished tars are skipped =====
dev_ids = set(proto['dev'].filename)
for t in DEV_TARS:
    run_tar_job('dev', t, dev_ids)

dev_scores = merge_parts('dev', DEV_TARS, dev_ids, 'aasist_dev.tsv')
res = metrics(dev_scores, proto['dev'], 'dev_full')
first2000 = set(proto['dev'].filename.iloc[:2000])
metrics(dev_scores[dev_scores.filename.isin(first2000)], proto['dev'], 'dev_first2000')
print('\nReproduction check: the official repo reports ~15.2% EER, measured on only the first 2000 dev files.')
print('dev_first2000 should be close to that; dev_full will be somewhat different, which is expected.')
if res and res['EER_%'] > 50:
    print('!! EER above 50% means the scores are flipped (real/fake swapped). Stop and check.')

  flac_D_aa.tar: downloaded (6339 MB)          
  flac_D_aa.tar: 47400 audio files, scoring 47400


  0%|          | 0/741 [00:00<?, ?it/s]

  saved dev_flac_D_aa.tsv to Drive
  flac_D_ab.tar: downloaded (6333 MB)          
  flac_D_ab.tar: 47400 audio files, scoring 47400


  0%|          | 0/741 [00:00<?, ?it/s]

  saved dev_flac_D_ab.tsv to Drive
  flac_D_ac.tar: downloaded (6457 MB)          
  flac_D_ac.tar: 47334 audio files, scoring 46150


  0%|          | 0/722 [00:00<?, ?it/s]

  saved dev_flac_D_ac.tsv to Drive
dev: scored 140950 of 140950 expected files; missing 0
{'set': 'dev_full', 'n_files': 140950, 'n_bonafide': 31334, 'minDCF': 0.3148, 'EER_%': 15.2, 'CLLR': 3.4049, 'actDCF': 0.9755}
{'set': 'dev_first2000', 'n_files': 2000, 'n_bonafide': 480, 'minDCF': 0.2659, 'EER_%': 13.75, 'CLLR': 3.2916, 'actDCF': 0.9743}

Reproduction check: the official repo reports ~15.2% EER, measured on only the first 2000 dev files.
dev_first2000 should be close to that; dev_full will be somewhat different, which is expected.


In [ ]:
# ===== CELL 5: Score the EVAL set (10 tars, ~94 GB, one at a time). Rerun if it stops =====
eval_ids = set(proto['eval'].filename)
for t in EVAL_TARS:
    run_tar_job('eval', t, eval_ids)

eval_scores = merge_parts('eval', EVAL_TARS, eval_ids, 'aasist_eval.tsv')
metrics(eval_scores, proto['eval'], 'eval_pooled')
print('Pooled eval only. The per-shift slices are built by the evaluation harness from aasist_eval.tsv.')

[skip] flac_E_aa.tar already scored for eval
[skip] flac_E_ab.tar already scored for eval
[skip] flac_E_ac.tar already scored for eval
[skip] flac_E_ad.tar already scored for eval
[skip] flac_E_ae.tar already scored for eval
[skip] flac_E_af.tar already scored for eval
[skip] flac_E_ag.tar already scored for eval
[skip] flac_E_ah.tar already scored for eval
  flac_E_ai.tar: downloaded (8072 MB)          
  flac_E_ai.tar: 68188 audio files, scoring 68188


  0%|          | 0/1066 [00:00<?, ?it/s]

  saved eval_flac_E_ai.tsv to Drive
  flac_E_aj.tar: downloaded (8223 MB)          
  flac_E_aj.tar: 68180 audio files, scoring 67082


  0%|          | 0/1049 [00:00<?, ?it/s]

  saved eval_flac_E_aj.tsv to Drive
eval: scored 680774 of 680774 expected files; missing 0
{'set': 'eval_pooled', 'n_files': 680774, 'n_bonafide': 138688, 'minDCF': 0.6663, 'EER_%': 28.11, 'CLLR': 4.3713, 'actDCF': 0.9715}
Pooled eval only. The per-shift slices are built by the evaluation harness from aasist_eval.tsv.


In [ ]:
# # edit this later
# # ===== CELL 6: DFADD + VCTK bona fide: prepare the test set (16 kHz FLAC) and score it =====
# DFADD_ZIPS = {'GradTTS': 'DATASET_GradTTS.zip', 'NaturalSpeech2': 'DATASET_NaturalSpeech2.zip',
#               'StyleTTS2': 'DATASET_StyleTTS2.zip', 'MatchaTTS': 'DATASET_MatchaTTS.zip',
#               'PflowTTS': 'DATASET_PflowTTS.zip', 'bonafide': 'DATASET_VCTK_BONAFIDE.zip'}
# DFADD_DIR = '/content/dfadd_test_16k'
# DFADD_TAR_ON_DRIVE = os.path.join(OUT_DIR, 'dfadd', 'dfadd_test_16k.tar')
# TEST_FOLDER = re.compile(r'(^|/)(test|testing)(/|_|$)', re.I)
# TEST_SPEAKERS = re.compile(r'p22[78]')      # DFADD's test speakers are p227 and p228
# os.makedirs(DFADD_DIR, exist_ok=True)

# if os.path.exists(DFADD_TAR_ON_DRIVE) and not find_audio(DFADD_DIR):
#     extract_tar(DFADD_TAR_ON_DRIVE, '/content')          # already prepared in an earlier session

# for gen, zname in DFADD_ZIPS.items():
#     flag = os.path.join(DFADD_DIR, f'.{gen}.done')
#     if os.path.exists(flag):
#         print(f'[skip] {gen} already prepared')
#         continue
#     zpath = download(f'{DFADD_URL}/{zname}', os.path.join(WORK_DIR, zname))
#     with zipfile.ZipFile(zpath) as z:
#         audio = [n for n in z.namelist() if n.lower().endswith(('.wav', '.flac'))]
#         test = [n for n in audio if TEST_FOLDER.search(n)] or [n for n in audio if TEST_SPEAKERS.search(n)]
#         top = sorted({'/'.join(n.split('/')[:2]) for n in audio})[:15]
#         print(f'{gen}: {len(audio)} audio files in zip, {len(test)} test files')
#         assert test, f'No test files found in {zname}. Folders inside: {top}'
#         for n in tqdm(test, leave=False):
#             x, sr = sf.read(io.BytesIO(z.read(n)), dtype='float32')
#             if x.ndim > 1:
#                 x = x.mean(axis=1)
#             if sr != 16000:
#                 g = math.gcd(sr, 16000)
#                 x = resample_poly(x, 16000 // g, sr // g).astype('float32')
#             file_id = gen + '__' + re.sub(r'[^A-Za-z0-9_.-]', '_', os.path.splitext(n)[0])  # unique across generators
#             sf.write(os.path.join(DFADD_DIR, file_id + '.flac'), np.clip(x, -1, 1), 16000, subtype='PCM_16')
#     os.remove(zpath)
#     open(flag, 'w').close()

# dfadd_audio = find_audio(DFADD_DIR)
# rows = []
# for fid in sorted(dfadd_audio):
#     gen = fid.split('__')[0]
#     spk = re.search(r'p\d{3}', fid)
#     rows.append({'SPEAKER_ID': spk.group(0) if spk else '-', 'filename': fid,
#                  'GENERATOR': '-' if gen == 'bonafide' else gen,
#                  'KEY': 'bonafide' if gen == 'bonafide' else 'spoof'})
# dfadd_proto = pd.DataFrame(rows)
# save_tsv(dfadd_proto, os.path.join(OUT_DIR, 'dfadd', 'dfadd_test_protocol.tsv'))
# print(dfadd_proto.groupby(['KEY', 'GENERATOR']).size().to_string())
# print('Speakers found:', sorted(dfadd_proto.SPEAKER_ID.unique()))

# dfadd_scores = score(dfadd_proto.filename.tolist(), [dfadd_audio[f] for f in dfadd_proto.filename])
# save_tsv(dfadd_scores, os.path.join(OUT_DIR, 'scores', 'aasist_dfadd.tsv'))

# bona = dfadd_proto[dfadd_proto.KEY == 'bonafide']
# for gen in sorted(g for g in dfadd_proto.GENERATOR.unique() if g != '-'):
#     subset = pd.concat([bona, dfadd_proto[dfadd_proto.GENERATOR == gen]])
#     metrics(dfadd_scores[dfadd_scores.filename.isin(subset.filename)], subset, f'dfadd_{gen}')
# metrics(dfadd_scores, dfadd_proto, 'dfadd_pooled')

# # Bona fide check: DFADD's real speech is VCTK, not LibriVox, so compare real-speech scores separately
# dfadd_bona = dfadd_scores[dfadd_scores.filename.isin(bona.filename)]['cm-score']
# print(f'Median bona fide score  DFADD/VCTK: {dfadd_bona.median():.2f}')
# if 'dev_scores' in globals():
#     dev_bona = dev_scores[dev_scores.filename.isin(proto['dev'][proto['dev'].KEY == 'bonafide'].filename)]['cm-score']
#     print(f'Median bona fide score  ASVspoof 5 dev: {dev_bona.median():.2f}  (a big gap = real VCTK speech gets rejected)')

# if not os.path.exists(DFADD_TAR_ON_DRIVE):
#     subprocess.run(['tar', '-cf', DFADD_TAR_ON_DRIVE, '-C', '/content', os.path.basename(DFADD_DIR)], check=True)
#     print('Prepared DFADD test audio saved to Drive for the team:', DFADD_TAR_ON_DRIVE)

# ===== CELL 6: DFADD + VCTK bona fide: prepare the test set (16 kHz FLAC) and score it =====
DFADD_ZIPS = {'GradTTS': 'DATASET_GradTTS.zip', 'NaturalSpeech2': 'DATASET_NaturalSpeech2.zip',
              'StyleTTS2': 'DATASET_StyleTTS2.zip', 'MatchaTTS': 'DATASET_MatchaTTS.zip',
              'PflowTTS': 'DATASET_PflowTTS.zip', 'bonafide': 'DATASET_VCTK_BONAFIDE.zip'}
DFADD_DIR = '/content/dfadd_test_16k'
DFADD_TAR_ON_DRIVE = os.path.join(OUT_DIR, 'dfadd', 'dfadd_test_16k.tar')
PREP_LOG = os.path.join(OUT_DIR, 'dfadd', 'dfadd_prep_log.txt')    # how each zip's test files were chosen
SUMMARY = os.path.join(OUT_DIR, 'dfadd', 'dfadd_summary.txt')      # counts, metrics, bona fide medians (for the report)
TEST_FOLDER = re.compile(r'(^|/)(test|testing)(/|_|$)', re.I)
TEST_SPEAKERS = re.compile(r'p22[78]')
EXPECTED_TEST_SPEAKERS = ['p227', 'p228']   # DFADD paper: test split = speakers p227 and p228
os.makedirs(DFADD_DIR, exist_ok=True)


def speakers_in(names):
    return sorted({m.group(0) for n in names for m in [re.search(r'p\d{3}', n)] if m})


if os.path.exists(DFADD_TAR_ON_DRIVE) and not find_audio(DFADD_DIR):
    extract_tar(DFADD_TAR_ON_DRIVE, '/content')          # already prepared in an earlier session

for gen, zname in DFADD_ZIPS.items():
    flag = os.path.join(DFADD_DIR, f'.{gen}.done')
    if os.path.exists(flag):
        print(f'[skip] {gen} already prepared')
        continue
    zpath = download(f'{DFADD_URL}/{zname}', os.path.join(WORK_DIR, zname))
    with zipfile.ZipFile(zpath) as z:
        audio = [n for n in z.namelist() if n.lower().endswith(('.wav', '.flac'))]
        by_folder = [n for n in audio if TEST_FOLDER.search(n)]
        by_speaker = [n for n in audio if TEST_SPEAKERS.search(n)]
        test = by_folder or by_speaker
        top = sorted({'/'.join(n.split('/')[:2]) for n in audio})[:15]
        lines = [f'{gen}: {len(audio)} audio files in zip',
                 f'  files in a test folder: {len(by_folder)} (speakers {speakers_in(by_folder)})',
                 f'  files from p227/p228:   {len(by_speaker)}',
                 f'  used: {"test folder" if by_folder else "speakers p227/p228"} -> {len(test)} files, speakers {speakers_in(test)}',
                 f'  both rules pick the same files: {set(by_folder) == set(by_speaker)}']
        if speakers_in(test) != EXPECTED_TEST_SPEAKERS:
            lines.append(f'  WARNING: expected test speakers {EXPECTED_TEST_SPEAKERS}. Check before using DFADD results.')
        print('\n'.join(lines))
        open(PREP_LOG, 'a').write('\n'.join(lines) + '\n')
        assert test, f'No test files found in {zname}. Folders inside: {top}'
        for n in tqdm(test, leave=False):
            x, sr = sf.read(io.BytesIO(z.read(n)), dtype='float32')
            if x.ndim > 1:
                x = x.mean(axis=1)
            if sr != 16000:
                g = math.gcd(sr, 16000)
                x = resample_poly(x, 16000 // g, sr // g).astype('float32')
            file_id = gen + '__' + re.sub(r'[^A-Za-z0-9_.-]', '_', os.path.splitext(n)[0])  # unique across generators
            sf.write(os.path.join(DFADD_DIR, file_id + '.flac'), np.clip(x, -1, 1), 16000, subtype='PCM_16')
    os.remove(zpath)
    open(flag, 'w').close()

dfadd_audio = find_audio(DFADD_DIR)
rows = []
for fid in sorted(dfadd_audio):
    gen = fid.split('__')[0]
    spk = re.search(r'p\d{3}', fid)
    rows.append({'SPEAKER_ID': spk.group(0) if spk else '-', 'filename': fid,
                 'GENERATOR': 'bonafide' if gen == 'bonafide' else gen,
                 'KEY': 'bonafide' if gen == 'bonafide' else 'spoof'})
dfadd_proto = pd.DataFrame(rows)
save_tsv(dfadd_proto, os.path.join(OUT_DIR, 'dfadd', 'dfadd_test_protocol.tsv'))

summary = ['DFADD test set: file counts per generator and speaker',
           pd.crosstab(dfadd_proto.GENERATOR, dfadd_proto.SPEAKER_ID, margins=True, margins_name='total').to_string(), '']
found = sorted(s for s in dfadd_proto.SPEAKER_ID.unique() if s != '-')
summary.append(f'Speakers: {found}  ' + ('(matches the DFADD test split)' if found == EXPECTED_TEST_SPEAKERS
                                          else f'!! expected {EXPECTED_TEST_SPEAKERS}'))

dfadd_scores = score(dfadd_proto.filename.tolist(), [dfadd_audio[f] for f in dfadd_proto.filename])
save_tsv(dfadd_scores, os.path.join(OUT_DIR, 'scores', 'aasist_dfadd.tsv'))

summary += ['', 'AASIST metrics (bona fide = all VCTK test files vs. each generator)']
bona = dfadd_proto[dfadd_proto.KEY == 'bonafide']
for gen in sorted(g for g in dfadd_proto.GENERATOR.unique() if g != 'bonafide'):
    subset = pd.concat([bona, dfadd_proto[dfadd_proto.GENERATOR == gen]])
    summary.append(str(metrics(dfadd_scores[dfadd_scores.filename.isin(subset.filename)], subset, f'dfadd_{gen}')))
summary.append(str(metrics(dfadd_scores, dfadd_proto, 'dfadd_pooled')))

# Bona fide check: DFADD's real speech is VCTK, not LibriVox, so compare real-speech scores separately
dev_file = os.path.join(OUT_DIR, 'scores', 'aasist_dev.tsv')
dev_all = pd.read_csv(dev_file, sep='\t', dtype={'filename': str})
dev_bona = dev_all[dev_all.filename.isin(proto['dev'][proto['dev'].KEY == 'bonafide'].filename)]['cm-score']
dfadd_bona = dfadd_scores[dfadd_scores.filename.isin(bona.filename)]['cm-score']
summary += ['', 'Bona fide (real speech) scores, higher = more real:',
            f'  DFADD / VCTK:     median {dfadd_bona.median():.2f}  (n={len(dfadd_bona)})',
            f'  ASVspoof 5 dev:   median {dev_bona.median():.2f}  (n={len(dev_bona)})',
            '  A large gap means AASIST rejects real VCTK speech, not just that it misses DFADD spoofs.']

print('\n'.join(summary))
open(SUMMARY, 'w').write('\n'.join(summary) + '\n')
print('\nSaved for the report:', SUMMARY)

if not os.path.exists(DFADD_TAR_ON_DRIVE):
    subprocess.run(['tar', '-cf', DFADD_TAR_ON_DRIVE, '-C', '/content', os.path.basename(DFADD_DIR)], check=True)
    print('Prepared DFADD test audio saved to Drive for the team:', DFADD_TAR_ON_DRIVE)

  DATASET_GradTTS.zip: downloaded (1741 MB)          
GradTTS: 32700 audio files in zip
  files in a test folder: 600 (speakers ['p227', 'p228'])
  files from p227/p228:   600
  used: test folder -> 600 files, speakers ['p227', 'p228']
  both rules pick the same files: True


  0%|          | 0/600 [00:00<?, ?it/s]

  DATASET_NaturalSpeech2.zip: downloaded (2707 MB)          
NaturalSpeech2: 32700 audio files in zip
  files in a test folder: 600 (speakers ['p227', 'p228'])
  files from p227/p228:   600
  used: test folder -> 600 files, speakers ['p227', 'p228']
  both rules pick the same files: True


  0%|          | 0/600 [00:00<?, ?it/s]

  DATASET_StyleTTS2.zip: downloaded (1922 MB)          
StyleTTS2: 32700 audio files in zip
  files in a test folder: 600 (speakers ['p227', 'p228'])
  files from p227/p228:   600
  used: test folder -> 600 files, speakers ['p227', 'p228']
  both rules pick the same files: True


  0%|          | 0/600 [00:00<?, ?it/s]

... (4 outputs left out)

  DATASET_VCTK_BONAFIDE.zip: downloaded (3485 MB)          
bonafide: 44455 audio files in zip
  files in a test folder: 755 (speakers ['p227', 'p228'])
  files from p227/p228:   755
  used: test folder -> 755 files, speakers ['p227', 'p228']
  both rules pick the same files: True


  0%|          | 0/755 [00:00<?, ?it/s]

  0%|          | 0/59 [00:00<?, ?it/s]

{'set': 'dfadd_GradTTS', 'n_files': 1355, 'n_bonafide': 755, 'minDCF': 0.2408, 'EER_%': 8.85, 'CLLR': 3.1116, 'actDCF': 0.985}
{'set': 'dfadd_MatchaTTS', 'n_files': 1355, 'n_bonafide': 755, 'minDCF': 0.1061, 'EER_%': 4.14, 'CLLR': 2.0387, 'actDCF': 0.9167}
{'set': 'dfadd_NaturalSpeech2', 'n_files': 1355, 'n_bonafide': 755, 'minDCF': 0.005, 'EER_%': 0.37, 'CLLR': 0.0669, 'actDCF': 0.0583}
{'set': 'dfadd_PflowTTS', 'n_files': 1355, 'n_bonafide': 755, 'minDCF': 0.0561, 'EER_%': 1.99, 'CLLR': 1.4022, 'actDCF': 0.8433}
{'set': 'dfadd_StyleTTS2', 'n_files': 1355, 'n_bonafide': 755, 'minDCF': 0.369, 'EER_%': 12.99, 'CLLR': 3.5365, 'actDCF': 0.9983}
{'set': 'dfadd_pooled', 'n_files': 3755, 'n_bonafide': 755, 'minDCF': 0.2063, 'EER_%': 7.81, 'CLLR': 2.0312, 'actDCF': 0.7603}
DFADD test set: file counts per generator and speaker
SPEAKER_ID      p227  p228  total
GENERATOR                        
GradTTS          300   300    600
MatchaTTS        300   300    600
NaturalSpeech2   300   300    600

In [ ]:
# ===== CELL 7: 10% held-out training speakers (the in-distribution reference) + AASIST scores on them =====
# The split is made ONCE and then reused, so everyone uses the same speakers.
split_file = os.path.join(OUT_DIR, 'splits', 'train_heldout_speakers.txt')
if os.path.exists(split_file):
    heldout = open(split_file).read().split()
    print(f'Using existing split: {len(heldout)} held-out speakers')
else:
    speakers = proto['train'].drop_duplicates('SPEAKER_ID')[['SPEAKER_ID', 'SPEAKER_GENDER']].sort_values('SPEAKER_ID')
    rng = np.random.RandomState(258)
    heldout = []
    for _, grp in speakers.groupby('SPEAKER_GENDER'):          # 10% of each gender
        ids = grp.SPEAKER_ID.to_numpy()
        heldout += list(rng.permutation(ids)[:max(1, round(0.10 * len(ids)))])
    heldout = sorted(heldout)
    open(split_file, 'w').write('\n'.join(heldout) + '\n')
    kept = sorted(set(speakers.SPEAKER_ID) - set(heldout))
    open(os.path.join(OUT_DIR, 'splits', 'train_kept_speakers.txt'), 'w').write('\n'.join(kept) + '\n')
    print(f'Created split: {len(heldout)} held-out speakers, {len(kept)} kept for training')

ref = proto['train'][proto['train'].SPEAKER_ID.isin(heldout)]
save_tsv(ref[['SPEAKER_ID', 'filename', 'ATTACK_LABEL', 'KEY']],
         os.path.join(OUT_DIR, 'splits', 'train_heldout_files.tsv'))
print(f'Held-out reference: {len(ref)} files ({(ref.KEY == "bonafide").sum()} bona fide)')

ref_ids = set(ref.filename)
for t in TRAIN_TARS:
    run_tar_job('trainref', t, ref_ids)
ref_scores = merge_parts('trainref', TRAIN_TARS, ref_ids, 'aasist_train_heldout_ref.tsv')
metrics(ref_scores, ref, 'train_heldout_ref')
print('Note: the released AASIST was trained on ALL training speakers, so this reference is optimistic for AASIST.')

Using existing split: 40 held-out speakers
Held-out reference: 18583 files (1903 bona fide)
[skip] flac_T_aa.tar already scored for trainref
[skip] flac_T_ab.tar already scored for trainref
[skip] flac_T_ac.tar already scored for trainref
  flac_T_ad.tar: downloaded (7161 MB)          
  flac_T_ad.tar: 36500 audio files, scoring 3629


  0%|          | 0/57 [00:00<?, ?it/s]

  saved trainref_flac_T_ad.tsv to Drive
  flac_T_ae.tar: downloaded (7135 MB)          
  flac_T_ae.tar: 36357 audio files, scoring 3709


  0%|          | 0/58 [00:00<?, ?it/s]

  saved trainref_flac_T_ae.tsv to Drive
trainref: scored 18583 of 18583 expected files; missing 0
{'set': 'train_heldout_ref', 'n_files': 18583, 'n_bonafide': 1903, 'minDCF': 0.001, 'EER_%': 0.05, 'CLLR': 0.008, 'actDCF': 0.0016}
Note: the released AASIST was trained on ALL training speakers, so this reference is optimistic for AASIST.


In [ ]:
# ===== CELL 8: Summary + README for the team =====
allm = json.load(open(METRICS_FILE)) if os.path.exists(METRICS_FILE) else {}
print(pd.DataFrame(allm.values()).to_string(index=False) if allm else 'No metrics yet.')

dfadd_done = os.path.exists(os.path.join(OUT_DIR, 'scores', 'aasist_dfadd.tsv'))
pending = '' if dfadd_done else '   [PENDING: Cell 6 not run yet]'
readme = f"""AASIST (official ASVspoof 5 checkpoint) outputs - Shriya

scores/aasist_dev.tsv, aasist_eval.tsv, aasist_train_heldout_ref.tsv, aasist_dfadd.tsv{pending}
    filename <tab> cm-score <tab> logit_spoof <tab> logit_bonafide
    cm-score = bona fide logit (official convention, higher = more likely real).
    Readable by the official evaluation package: python evaluation.py --m t1 --cm <scores> --cm_keys <keys>
keys/{{train,dev,eval}}_cm_keys.tsv    filename <tab> cm-label, official format
keys/ASVspoof5.eval.track_1.tsv      full eval protocol (codec, CODEC_SEED, attack) for building slices
splits/train_heldout_speakers.txt    the 10% held-out training speakers (seed 258, stratified by gender)
splits/train_kept_speakers.txt       the other 90% (use these to train the SSL head)
splits/train_heldout_files.tsv       every training file from the held-out speakers
metrics/aasist_metrics.json          official metrics (minDCF, EER, CLLR, actDCF) per set
dfadd/dfadd_test_protocol.tsv        SPEAKER_ID, filename, GENERATOR, KEY for the DFADD test set{pending}
dfadd/dfadd_test_16k.tar             prepared DFADD test audio, 16 kHz mono FLAC (bona fide = VCTK){pending}
dfadd/dfadd_summary.txt              counts per generator/speaker, metrics, bona fide medians{pending}
dfadd/dfadd_prep_log.txt             how each DFADD zip's test files were selected{pending}

Notes
- Model input: first 64,600 samples, short clips repeated (same as the official AASIST code).
- Reproduction: dev_full EER 15.20% matches the official repo's 15.2%. Ignore dev_first2000; the official
  code's first 2000 files come from an older metadata file, so it is not the same subset.
- train_heldout_ref (0.05% EER) is NOT held out for AASIST: the released checkpoint was trained on all
  training speakers. Do not use it as AASIST's in-distribution reference. It is the right reference for
  the SSL head, which never sees these speakers.
- Scores are raw, uncalibrated logits, so actDCF and CLLR look poor everywhere. Report minDCF and EER;
  report actDCF only after temperature scaling is fitted on dev.
"""
open(os.path.join(OUT_DIR, 'README.txt'), 'w').write(readme)

print('\nFiles in', OUT_DIR)
for dirpath, _, files in sorted(os.walk(OUT_DIR)):
    if '/parts' in dirpath:
        continue
    for fn in sorted(files):
        p = os.path.join(dirpath, fn)
        print(f'  {os.path.relpath(p, OUT_DIR):45s} {os.path.getsize(p) / 1e6:9.1f} MB')

                 set  n_files  n_bonafide  minDCF  EER_%   CLLR  actDCF
            dev_full   140950       31334  0.3148  15.20 3.4049  0.9755
       dev_first2000     2000         480  0.2659  13.75 3.2916  0.9743
         eval_pooled   680774      138688  0.6663  28.11 4.3713  0.9715
   train_heldout_ref    18583        1903  0.0010   0.05 0.0080  0.0016
       dfadd_GradTTS     1355         755  0.2408   8.85 3.1116  0.9850
     dfadd_MatchaTTS     1355         755  0.1061   4.14 2.0387  0.9167
dfadd_NaturalSpeech2     1355         755  0.0050   0.37 0.0669  0.0583
      dfadd_PflowTTS     1355         755  0.0561   1.99 1.4022  0.8433
     dfadd_StyleTTS2     1355         755  0.3690  12.99 3.5365  0.9983
        dfadd_pooled     3755         755  0.2063   7.81 2.0312  0.7603

Files in /content/drive/MyDrive/CMPE258_Deepfake/shriya_aasist
  README.txt                                          0.0 MB
  dfadd/dfadd_prep_log.txt                            0.0 MB
  dfadd/dfadd_summary.